# LLM Fine-Tuning Deep Dive, Part 2 of 3: What Must Riverside Pay to Change?

Part 1 decided **what behavior to practice**. Part 2 asks a different question: **how much model state must Riverside update, keep in memory, and store for each adaptation?**

Imagine Riverside wants separate local assistants for fiction editing, marketing campaigns, and author-specific personas. If every adaptation becomes another complete model, the technical choice quickly becomes an operating bill:

- more trainable state during every job;
- more optimizer memory;
- another large artifact to store and move; and
- fewer adaptations that fit on available hardware.

This notebook follows Riverside through four attempts to shrink that bill without changing the learning objective:

1. **Full fine-tuning:** let every weight move and expose the maximum bill.
2. **Partial freezing:** reuse most weights and update only a selected slice.
3. **LoRA:** keep one shared base and store a small correction per Riverside job.
4. **QLoRA:** compact the frozen base too when resident memory becomes the blocker.

Each exercise follows the same rhythm:

> Riverside bill → predict what can shrink → inspect one model object → run the mechanism → state what bill remains

The predictions concern physical training state, not model quality. Evaluation begins in Part 3.

## Three Questions to Carry Through Every Exercise

1. Which tensors receive gradients?
2. Which tensors still participate in the forward pass?
3. What must Riverside save to reconstruct this adaptation?

## Hardware Boundary

The code reuses the shared SmolLM2 profile: 135M on CPU, 360M on ordinary CUDA, and 1.7B on CUDA with at least 64 GiB. The small CPU path is enough to inspect every mechanism even when generated prose remains unremarkable.

## The Budget Story: Remove One Bottleneck at a Time

Riverside keeps the learning objective fixed and changes only where trainable state lives and how large it is.

```mermaid
flowchart LR
    F["Full FT\nall weights learn"] -->|"all gradients and optimizer state"| P["Partial freezing\nselected layers learn"]
    P -->|"still saves a full changed model"| L["LoRA\nsmall adapter learns"]
    L -->|"base still occupies memory"| Q["QLoRA path\ncompact frozen base + adapter"]
    Q --> E["Part 3\nevaluation and selection"]
```

Read the arrows as discovery order, not a required production pipeline.

| Strategy | Cost it removes | Cost it leaves |
| --- | --- | --- |
| Partial freezing | Optimizer state for frozen layers | Manual layer choice and a complete changed model |
| LoRA | Full per-job weight updates and checkpoints | A full-precision frozen base in memory |
| QLoRA | Much of the frozen base's memory footprint | Low-bit kernel and hardware constraints |

This notebook inspects trainability masks, parameter counts, matrix shapes, correction paths, and artifact contents. It does not rank resulting models.

![Parameter efficiency spectrum comparing full fine-tuning, partial fine-tuning, LoRA, and QLoRA](images/parameter-strategies-spectrum.png)

> **PyTorch → Keras:** `AutoModelForCausalLM.from_pretrained(...).to(device)` loads pretrained weights and moves the model to the training device (CPU/GPU); `model.generate()` inside `torch.no_grad()` runs autoregressive decoding without tracking gradients. **Keras/TF equivalent:** `TFAutoModelForCausalLM.from_pretrained(...)` loads weights, with `tf.device(...)` selecting the device; Keras has no built-in `.generate()` for causal LMs outside HF's `TFGenerationMixin.generate()`, and gradient tracking is simply skipped by not wrapping calls in a `tf.GradientTape()`.

In [ ]:
# Re-establish Part 1's hardware-aware SmolLM2 profile.
from pathlib import Path
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

CUDA_AVAILABLE = torch.cuda.is_available()
GPU_MEMORY_GIB = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3 if CUDA_AVAILABLE else 0.0
)

if not CUDA_AVAILABLE:
    MODEL_PROFILE = "cpu-small-135m"
    INSTRUCT_MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
    INSTRUCT_MODEL_REVISION = "12fd25f77366fa6b3b4b768ec3050bf629380bac"
elif GPU_MEMORY_GIB < 64:
    MODEL_PROFILE = "gpu-balanced-360m"
    INSTRUCT_MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
    INSTRUCT_MODEL_REVISION = "a10cc1512eabd3dde888204e902eca88bddb4951"
else:
    MODEL_PROFILE = "gpu-quality-1.7b"
    INSTRUCT_MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
    INSTRUCT_MODEL_REVISION = "31b70e2e869a7173562077fd711b654946d38674"

MODEL_NAME = INSTRUCT_MODEL_NAME
MODEL_REVISION = INSTRUCT_MODEL_REVISION
SYSTEM_PROMPT = "You are Riverside House's concise fiction-writing assistant."
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
DEMO_TRAIN_STEPS = 10
STORY_SEED = "Aria Voss stared at the signal counting itself out in prime numbers and"
PROMPT = "Continue this fiction narrative in the same style: " + STORY_SEED

device = "cuda" if CUDA_AVAILABLE else "cpu"
print(f"Using device: {device} ({MODEL_PROFILE}, {GPU_MEMORY_GIB:.1f} GiB CUDA memory)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
).to(device)
model_config = base_model.config
print(
    f"Loaded {MODEL_NAME}@{MODEL_REVISION[:8]}: {len(base_model.model.layers)} decoder layers, "
    f"hidden size {model_config.hidden_size}, "
    f"{sum(parameter.numel() for parameter in base_model.parameters()):,} parameters"
)
if not CUDA_AVAILABLE:
    print(
        "CPU disclaimer: the 135M checkpoint is large enough to expose freezing, LoRA, and "
        "checkpoint mechanics, but short runs may produce generic or unchanged text. Limited "
        "capacity and ten updates are expected constraints, not evidence against the techniques."
    )


def format_instruction(prompt, tokenizer_obj=tokenizer):
    """Render the native SmolLM2 chat contract used throughout the arc."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt.strip()},
    ]
    return tokenizer_obj.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def generate(model, prompt, max_new_tokens=60, instruction=True, tokenizer_obj=tokenizer):
    """Generate only new tokens using the shared instruction format when requested."""
    model.eval()
    model_input = format_instruction(prompt, tokenizer_obj) if instruction else prompt
    model_device = next(model.parameters()).device
    inputs = tokenizer_obj(model_input, return_tensors="pt")
    inputs = {name: tensor.to(model_device) for name, tensor in inputs.items()}
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            pad_token_id=tokenizer_obj.pad_token_id,
            eos_token_id=tokenizer_obj.eos_token_id,
        )
    completion = tokenizer_obj.decode(
        output[0][prompt_len:], skip_special_tokens=True
    ).strip()
    return completion or "[model stopped immediately]"


print(f"Baseline completion (sanity check): {generate(base_model, PROMPT)}")

In [ ]:
# Corpus loader, profile-specific artifact paths, and training/visualization imports.
try:
    _notebook_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]
except NameError:
    try:
        _notebook_dir = Path(__file__).parent
    except NameError:
        _notebook_dir = Path.cwd()

REPO_ROOT = _notebook_dir.parents[2]
CHECKPOINT_ROOT = REPO_ROOT / "checkpoints"
CHECKPOINT_DIR = CHECKPOINT_ROOT / "llm-finetuning" / MODEL_PROFILE
CONTENT_DIR = REPO_ROOT / "learning" / "genai" / "content"
if not CONTENT_DIR.exists():
    raise FileNotFoundError(
        f"Shared Riverside content directory not found at {CONTENT_DIR.absolute()}. "
        "Expected the GenAI-wide corpus under learning/genai/content."
    )

print(f"Shared content directory: {CONTENT_DIR.absolute()}")
print(f"Profile checkpoints: {CHECKPOINT_DIR.absolute()}")

NOVELS = {
    "scifi": "the-weight-of-distant-light",
    "fantasy": "the-tidebound-accord",
    "mystery": "the-cartographers-cipher",
    "historical": "the-silk-merchants-daughter",
    "cyberpunk": "neural-drift",
    "horror": "the-hollow-beneath",
    "literary": "the-weight-of-tides",
}


def load_corpus_paragraphs(novels=None, min_len=200):
    """Load qualifying paragraphs from every chapter of the selected novels."""
    if novels is None:
        novels = list(NOVELS.keys())
    paragraphs = []
    for alias in novels:
        novel_dir = NOVELS.get(alias)
        if not novel_dir:
            continue
        novel_path = CONTENT_DIR / novel_dir
        if not novel_path.exists():
            continue
        for path in sorted(novel_path.glob("chapter-*.txt")):
            text = path.read_text(encoding="utf-8")
            for paragraph in text.split("\n\n"):
                paragraph = paragraph.strip().replace("\n", " ")
                if len(paragraph) >= min_len:
                    paragraphs.append(paragraph)
    return paragraphs


def tokenize_causal(examples, tokenizer_obj, max_length=64):
    """Tokenize batched text into fixed-length causal language-modeling chunks."""
    tokens = tokenizer_obj(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_overflowing_tokens=True,
    )
    tokens["labels"] = [
        [(token if mask == 1 else -100) for token, mask in zip(ids, attention)]
        for ids, attention in zip(tokens["input_ids"], tokens["attention_mask"])
    ]
    return tokens


import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
from datasets import Dataset
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})
sns.set_theme(style="whitegrid", palette="muted")

print("Corpus loader, tokenize_causal(), and training/visualization imports ready.")


### Reloading Part 1's Instruction-Tuned LoRA Adapter

The later introspection section uses the instruction-tuning adapter produced by Part 1. The base model, tokenizer, target modules, tensor shapes, and hardware-selected profile must match exactly.

Artifacts live under `checkpoints/llm-finetuning/<profile>/`. Run Part 1 on the same hardware profile before this reload. The compatibility check below stops rather than attaching an adapter from another model size.

On CPU, the 135M adapter may still generate generic or unchanged text after the ten-step teaching run. That is expected because model capacity and optimization time are deliberately small; the adapter structure, trainable-parameter count, and checkpoint contract remain valid learning evidence.

> **PyTorch → Keras:** `PeftModel.from_pretrained(base_model, adapter_dir)` attaches previously trained LoRA adapter weights on top of a frozen base model, reconstructing the exact adapted model from Part 1's checkpoint. **Keras/TF equivalent:** there's no first-party Keras LoRA/PEFT library — the closest analog is reloading a full model (or a frozen-base-plus-trainable-sublayer subclass) via `model.load_weights(...)`; Keras users would typically just reload the entire fine-tuned model rather than a small swappable adapter.

In [ ]:
# Reload the instruction-tuned adapter only after Part 1 has regenerated it for MODEL_NAME.
instruction_adapter_dir = CHECKPOINT_DIR / "instruction-lora"
adapter_config_path = instruction_adapter_dir / "adapter_config.json"
if not adapter_config_path.is_file():
    raise FileNotFoundError(
        f"Missing {adapter_config_path}. Rerun Part 1 with MODEL_NAME={MODEL_NAME!r}."
    )

saved_adapter_config = json.loads(adapter_config_path.read_text(encoding="utf-8"))
saved_base = saved_adapter_config.get("base_model_name_or_path")
if saved_base != MODEL_NAME:
    raise RuntimeError(
        f"Checkpoint base {saved_base!r} is incompatible with {MODEL_NAME!r}. "
        "Leave the old artifact untouched and regenerate Part 1 checkpoints."
    )

instruct_base_reload = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
instruct_lora_model = PeftModel.from_pretrained(
    instruct_base_reload, instruction_adapter_dir
).to(device)
instruct_lora_model.eval()
print("Reloaded the SmolLM2 instruction adapter generated by Part 1.")
print(generate(instruct_lora_model, PROMPT))


---

## Concept 4: Full Fine-Tuning - Pay for Maximum Freedom

### The Riverside Bill

Full fine-tuning gives every weight permission to change. That is the least constrained way to adapt a model, but it also means every Riverside job pays for gradients and optimizer state across the entire network and saves another complete checkpoint.

### Make a Prediction

Before counting anything:

- trainable parameters should equal total parameters;
- every transformer block should remain writable; and
- the saved artifact must be a complete model, not a small Riverside-specific add-on.

### Inspect One Model

The next cell asks a deliberately simple question: if Riverside changes nothing about trainability, what percentage of the model joins the update? The answer establishes the physical reference for every cheaper strategy that follows.

> **Next bill to attack:** Riverside probably does not need to relearn general language for every house adaptation. Can it stop paying update-state costs for reusable layers?

> **PyTorch → Keras:** `p.numel()` counts elements in each parameter tensor and `p.requires_grad` flags whether it receives gradient updates; summing over `model.parameters()` gives total vs. trainable parameter counts. **Keras/TF equivalent:** `model.count_params()` gives total parameters directly, and the trainable subset is `sum(np.prod(w.shape) for w in model.trainable_weights)` — Keras tracks trainable/non-trainable via each layer's `trainable` attribute rather than a per-tensor `requires_grad` flag.

In [ ]:
# Inspect the unconstrained reference before trying to reduce it.
param_check_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
total_parameters = sum(parameter.numel() for parameter in param_check_model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in param_check_model.parameters()
    if parameter.requires_grad
)
trainable_percent = 100 * trainable_parameters / total_parameters
assert trainable_parameters == total_parameters

print("INSPECT FULL FINE-TUNING")
print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")
print(f"Trainable share: {trainable_percent:.1f}%")
print("Artifact shape: complete changed model")
print("Riverside takeaway: maximum update freedom also creates the maximum per-job state bill.")

del param_check_model

### Riverside Takeaway: What Full Fine-Tuning Costs

The prediction held structurally: every parameter joined the update, and Riverside would save another complete model for this adaptation.

Full fine-tuning buys maximum freedom. Its remaining problem is duplication: fiction, marketing, and author-persona jobs each become a large independent artifact.

## Concept 5: Partial Freezing - Reuse General Language

### The Riverside Bill

The base already knows general language. Riverside's first cost experiment freezes most transformer blocks and updates roughly the last quarter plus the final normalization layer.

### Make a Prediction

- early and middle blocks should have `requires_grad=False`;
- the final block slice should remain trainable;
- trainable state should shrink substantially; but
- the forward pass still uses the whole base, and the adaptation still saves a complete checkpoint.

The next figure turns that policy into a visible frozen/trainable map before the trainer runs.

### Inspect the Proposed Split Before Training

Riverside's policy is easy to say and easy to implement incorrectly: freeze everything, then re-enable roughly the last quarter of decoder blocks plus `model.norm`. The tied embedding/output weight stays frozen.

| Region | Prediction |
| --- | --- |
| Early and middle transformer blocks | Frozen |
| Last quarter of transformer blocks | Trainable |
| Final normalization | Trainable |
| Tied token embedding / output head | Frozen |

The visualization is a preflight check: if the colors do not match this prediction, Riverside should not spend compute on the run.

In [ ]:
# Optional visualization of the configured freezing policy; no gradient values are fabricated.
from transformers import AutoConfig
from matplotlib.patches import Patch

freeze_config = AutoConfig.from_pretrained(MODEL_NAME)
n_layers = freeze_config.num_hidden_layers
unfreeze_from = n_layers - max(2, n_layers // 4)
layers = [f"Block {index}" for index in range(n_layers)] + [
    "Final norm",
    "Tied embedding / output head",
]
trainable_mask = [index >= unfreeze_from for index in range(n_layers)] + [True, False]
colors = ["coral" if is_trainable else "lightblue" for is_trainable in trainable_mask]
positions = np.arange(len(layers))
tick_stride = max(1, n_layers // 12)
tick_positions = list(range(0, n_layers, tick_stride)) + [n_layers, n_layers + 1]

fig, axis = plt.subplots(figsize=(9, max(6, n_layers * 0.28)))
axis.barh(positions, np.ones(len(layers)), color=colors, edgecolor="black")
axis.set_yticks(tick_positions)
axis.set_yticklabels([layers[index] for index in tick_positions])
axis.set_xlim(0, 1)
axis.set_xticks([])
axis.invert_yaxis()
axis.set_title(
    f"Configured Partial-Freezing Policy ({n_layers} transformer blocks)",
    fontweight="bold",
)
axis.legend(
    handles=[
        Patch(facecolor="lightblue", edgecolor="black", label="Frozen"),
        Patch(facecolor="coral", edgecolor="black", label="Trainable"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=2,
)
plt.tight_layout()
plt.show()

frozen_blocks = unfreeze_from
trainable_blocks = n_layers - unfreeze_from
print(f"Frozen transformer blocks:    {frozen_blocks}/{n_layers}")
print(f"Trainable transformer blocks: {trainable_blocks}/{n_layers}")
print("Final norm:                   trainable")
print("Tied embedding/output head:   frozen")
print("This policy predicts trainable state, not gradient magnitude or model quality.")

> **PyTorch → Keras:** setting `param.requires_grad = False` on every parameter freezes the entire model so no gradients flow to it during backprop. **Keras/TF equivalent:** `layer.trainable = False` on each layer (or `model.trainable = False` for the whole model) before compiling — Keras freezes at layer granularity, not per-tensor, and the change only takes effect after the model is (re)compiled.

In [ ]:
# Load a fresh base model instance to selectively freeze/unfreeze
freeze_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

# Start every parameter frozen; the next cell re-enables gradients on the trainable slice
for param in freeze_model.parameters():
    param.requires_grad = False


### Selectively Unfreezing the Last ~25% of Blocks

Every parameter starts frozen above. This cell re-enables `requires_grad` only on the last few
`model.layers` blocks (`unfreeze_from` onward) plus `model.norm`, matching the split
visualized earlier. Because `lm_head.weight` is tied to `model.embed_tokens.weight`, both remain
frozen so the optimizer cannot silently update the token embedding through the output head.

> **PyTorch → Keras:** `model.named_parameters()` yields `(name, tensor)` pairs so individual parameters can be selectively re-enabled (`param.requires_grad = True`) by matching name substrings like block index or layer-norm/head names. **Keras/TF equivalent:** iterate `model.layers` and set `layer.trainable = True` for the specific layers you want to unfreeze (matched by `layer.name`), then recompile the model — Keras' selective-unfreezing story works the same way but at whole-layer granularity rather than per-parameter-tensor.

In [ ]:
n_layers = len(freeze_model.model.layers)
unfreeze_from = n_layers - max(2, n_layers // 4)

# Llama exposes decoder blocks as model.layers.
decoder_layers = freeze_model.model.layers
for layer in decoder_layers[unfreeze_from:]:
    for parameter in layer.parameters():
        parameter.requires_grad = True
for parameter in freeze_model.model.norm.parameters():
    parameter.requires_grad = True

# lm_head is tied to model.embed_tokens. Leaving it frozen avoids silently unfreezing embeddings.
assert freeze_model.lm_head.weight is freeze_model.model.embed_tokens.weight
assert not freeze_model.model.embed_tokens.weight.requires_grad

trainable_layer_prefixes = tuple(
    f"model.layers.{layer_index}." for layer_index in range(unfreeze_from, n_layers)
)
for name, parameter in freeze_model.named_parameters():
    if name.startswith(trainable_layer_prefixes) or name.startswith("model.norm."):
        assert parameter.requires_grad, f"Expected trainable Llama parameter: {name}"

trainable = sum(parameter.numel() for parameter in freeze_model.parameters() if parameter.requires_grad)
total = sum(parameter.numel() for parameter in freeze_model.parameters())
print(
    f"Partial fine-tuning: {trainable:,}/{total:,} parameters trainable "
    f"({trainable / total * 100:.2f}%)"
)
print("Tied token embedding/output head remains frozen.")

### Exercise - Train Only the Selected Slice

**Riverside problem:** the trainability map is configured, but the adaptation has not yet written anything.

**Prediction:** `Trainer` will traverse the complete model during the forward pass but update only tensors whose `requires_grad` flag is enabled. The resulting artifact will still be a complete checkpoint.

**Run:** reuse the causal-language-modeling dataset pattern, train the selected slice for ten steps, and save the full model.

> **PyTorch → Keras:** `Dataset`/`TrainingArguments`/`Trainer.train()` is HF's high-level PyTorch training loop (batching, optimizer, logging all handled internally), and `save_pretrained()` writes the model and config to disk. **Keras/TF equivalent:** `tf.data.Dataset` for batching, `model.compile(optimizer=..., loss=...)` to configure training, `model.fit(dataset, epochs=...)` to run it, and `model.save(...)` / `model.save_weights(...)` to persist — Keras' `fit()` plays the same role as `Trainer.train()`.

In [ ]:
# Run the same training objective while changing only which parameters may update.
freeze_dataset = Dataset.from_dict(
    {"text": load_corpus_paragraphs(novels=["fantasy", "cyberpunk"])}
)
freeze_tokenized = freeze_dataset.map(
    lambda examples: tokenize_causal(examples, tokenizer),
    batched=True,
    remove_columns=["text"],
)

freeze_trainable = sum(
    parameter.numel() for parameter in freeze_model.parameters()
    if parameter.requires_grad
)
freeze_total = sum(parameter.numel() for parameter in freeze_model.parameters())
print("BEFORE PARTIAL-FREEZING TRAINING")
print(f"Training chunks: {len(freeze_tokenized):,}")
print(f"Trainable share: {100 * freeze_trainable / freeze_total:.2f}%")
print("Forward pass: complete model")
print("Writable state: selected upper slice")

training_args_freeze = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR / "partial-freeze"),
    per_device_train_batch_size=1,
    max_steps=DEMO_TRAIN_STEPS,
    logging_steps=1,
    save_strategy="no",
    learning_rate=1e-4,
    report_to="none",
)
trainer_freeze = Trainer(
    model=freeze_model,
    args=training_args_freeze,
    train_dataset=freeze_tokenized,
)
trainer_freeze.train()
freeze_model.save_pretrained(str(CHECKPOINT_DIR / "partial-freeze"))

print("\nAFTER TRAINING")
print("Saved artifact: complete partial-freeze checkpoint")
print("Riverside takeaway: update state shrank, but per-job checkpoint duplication remains.")

## Concept 6: LoRA - Share One Base Across Riverside Jobs

### The Riverside Bill

Partial freezing reduced update state but still produced a complete checkpoint. If Riverside creates fiction, marketing, and author-persona adaptations, storing another full model per job remains the expensive part.

LoRA changes the unit of ownership: Riverside keeps one frozen base and learns a small correction for each job:

$$
\text{layer output}=Wx+B(Ax).
$$

$W$ is frozen; $BA$ is the learned correction. Rank $r$ is the bottleneck width and limits the independent directions that correction can express.

The projection width depends on the selected profile, so the notebook derives it from `base_model.config.hidden_size`. For a square width-$d$ projection, full updating exposes $d^2$ weights while a rank-8 adapter exposes $8d+d8=16d$ weights. The code below prints the actual values for 135M, 360M, or 1.7B instead of assuming a 960-wide model.

![A frozen query projection beside a rank-8 LoRA detour that compresses an activation to eight values, expands it to the runtime hidden width, and adds the correction](images/lora-low-rank-adaptation.png)

### Make a Prediction

- the base projection $W$ should remain frozen;
- only small $A$ and $B$ matrices should receive gradients;
- trainable state should collapse relative to full fine-tuning; and
- the saved Riverside artifact should be an adapter that still depends on the pinned base.

The next cells inspect one real projection, train the adapter, open its matrices, and swap two Riverside adapters onto one compatible base.

> **Next Riverside bill:** the adapter is small, but every job still needs the frozen base resident in memory. Can that base be stored more compactly?

> **PyTorch → Keras:** `LoraConfig(...)` declares the LoRA hyperparameters (rank, target modules, alpha), `get_peft_model(base, config)` wraps the frozen base model with trainable low-rank adapter matrices injected into the named target modules, and `print_trainable_parameters()` reports the resulting trainable/total ratio. **Keras/TF equivalent:** no first-party Keras LoRA API exists — the closest honest analog is manually freezing most layers (`layer.trainable = False`) and, for true low-rank adapters, hand-writing a custom `keras.layers.Layer` that adds a `BA` low-rank branch alongside a frozen dense layer; there's no drop-in `get_peft_model` equivalent.

In [ ]:
# Inspect the Riverside adapter before training it.
lora_config_pt = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
)

lora_pt_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
lora_pt_model = get_peft_model(lora_pt_base, lora_config_pt)
lora_trainable = sum(
    parameter.numel() for parameter in lora_pt_model.parameters() if parameter.requires_grad
)
lora_total = sum(parameter.numel() for parameter in lora_pt_model.parameters())

print("INSPECT THE LORA UPDATE PATH")
print(f"Target modules: {LORA_TARGET_MODULES}")
print(f"Adapter rank: {lora_config_pt.r}")
print(f"Trainable share: {100 * lora_trainable / lora_total:.4f}%")
print("Base model: frozen and shared")
print("Expected Riverside artifact: adapter only")

### Exercise 1 - Feed the Shared Objective Through the Adapter

**Riverside problem:** the adapter path exists, but it has not learned a job-specific correction.

**Prediction:** the dataset shape stays identical to causal training, while only LoRA tensors receive updates.

**Run:** build fixed-length chunks from complete selected novels, then send those chunks through the LoRA-wrapped model.

In [ ]:
# Load every qualifying paragraph from all chapters in the three selected genres.
lora_pt_paragraphs = load_corpus_paragraphs(
    novels=["mystery", "horror", "literary"]
)

lora_pt_dataset = Dataset.from_dict({"text": lora_pt_paragraphs})
lora_pt_tokenized = lora_pt_dataset.map(
    lambda examples: tokenize_causal(examples, tokenizer),
    batched=True,
    remove_columns=["text"],
)

print(
    f"LoRA corpus: {len(lora_pt_paragraphs):,} paragraphs -> "
    f"{len(lora_pt_tokenized):,} fixed-length training chunks"
)
print("This dataset supports the LoRA mechanism exercise; Part 3 owns behavioral comparison.")

### Exercise 2 - Save the Riverside Correction

**Prediction:** `save_pretrained()` should write small adapter files rather than another complete base checkpoint.

**Run:** train for ten steps, save the adapter, and compare its files with the resident base size in the portability exercise below.

> **PyTorch → Keras:** same `TrainingArguments`/`Trainer.train()`/`save_pretrained()` pattern as the earlier training cell, here training only the LoRA adapter's parameters (the base stays frozen) and saving just the small adapter weights. **Keras/TF equivalent:** `model.compile(...)` + `model.fit(...)` with only the adapter sublayer's `trainable = True`, then `model.save_weights(...)` — since Keras has no adapter abstraction, this would typically save the whole model rather than a separate small adapter file.

In [ ]:
# Train only the LoRA correction, then inspect the saved Riverside artifact.
training_args_lora_pt = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR / "peft-lora"),
    per_device_train_batch_size=1,
    max_steps=DEMO_TRAIN_STEPS,  # short instructional run; increase for a real convergence study
    logging_steps=1,
    save_strategy="no",
    learning_rate=2e-4,
    report_to="none",
)

# Wire the LoRA-wrapped model, its training args, and the tokenized dataset together
trainer_lora_pt = Trainer(
    model=lora_pt_model, args=training_args_lora_pt, train_dataset=lora_pt_tokenized
)

# Run training -- only the LoRA adapter matrices actually receive gradient updates
trainer_lora_pt.train()

# Persist just the small adapter weights (the frozen base isn't re-saved)
lora_artifact_dir = CHECKPOINT_DIR / "peft-lora"
lora_pt_model.save_pretrained(str(lora_artifact_dir))
lora_artifact_bytes = sum(
    path.stat().st_size for path in lora_artifact_dir.iterdir() if path.is_file()
)
print("RIVERSIDE LORA ARTIFACT")
print(f"Trainable values: {lora_trainable:,}")
print(f"Saved adapter files: {lora_artifact_bytes / 1024:.0f} KiB")
print("Frozen base files were not copied into this adapter directory.")


### Optional Depth: Visualize the Low-Rank Correction

The main practical path needs only the measured parameter count and a compatible adapter. The next visualization reads the selected SmolLM2 hidden width at runtime and draws its rank-8 $A$ and $B$ path.

Use this section when you want to see why the correction has rank at most $r$. Otherwise, skip to **Crack Open the Adapter We Just Trained**.

The code computes the full-projection and adapter counts from the loaded model, so the figure remains accurate for the 135M CPU, 360M GPU, and 1.7B high-memory GPU profiles.

In [ ]:
# Optional shape-and-count view of one SmolLM2 projection.
from matplotlib.patches import FancyBboxPatch

projection_width = base_model.config.hidden_size
adapter_rank = 8
full_projection_params = projection_width * projection_width
adapter_a_params = adapter_rank * projection_width
adapter_b_params = projection_width * adapter_rank
adapter_total_params = adapter_a_params + adapter_b_params
adapter_percent = adapter_total_params / full_projection_params * 100

fig, (count_axis, path_axis) = plt.subplots(1, 2, figsize=(13, 4.5))

labels = ["Full projection\nupdate", "LoRA A + B"]
counts = [full_projection_params, adapter_total_params]
count_axis.bar(labels, counts, color=["steelblue", "mediumseagreen"], edgecolor="black")
count_axis.set_yscale("log")
count_axis.set_ylabel("Trainable values (log scale)")
count_axis.set_title("One Projection: Full Update vs LoRA", fontweight="bold")
for index, count in enumerate(counts):
    count_axis.text(index, count * 1.08, f"{count:,}", ha="center", va="bottom")

path_axis.set_xlim(0, 10)
path_axis.set_ylim(0, 3)
path_axis.axis("off")
boxes = [
    (0.3, 1.0, 1.5, 1.0, f"input x\n{projection_width} dims", "lightblue"),
    (2.4, 1.0, 1.7, 1.0, f"A\n{adapter_rank} x {projection_width}", "mediumseagreen"),
    (4.8, 1.0, 1.2, 1.0, f"rank-{adapter_rank}\nbottleneck", "#f6d365"),
    (6.7, 1.0, 1.7, 1.0, f"B\n{projection_width} x {adapter_rank}", "coral"),
    (9.0, 1.0, 0.7, 1.0, "delta", "#c7a4e0"),
]
for x_pos, y_pos, width, height, label, color in boxes:
    path_axis.add_patch(
        FancyBboxPatch(
            (x_pos, y_pos),
            width,
            height,
            boxstyle="round,pad=0.05",
            facecolor=color,
            edgecolor="black",
        )
    )
    path_axis.text(x_pos + width / 2, y_pos + height / 2, label, ha="center", va="center")
for start, end in [(1.8, 2.4), (4.1, 4.8), (6.0, 6.7), (8.4, 9.0)]:
    path_axis.annotate("", xy=(end, 1.5), xytext=(start, 1.5), arrowprops={"arrowstyle": "->"})
path_axis.set_title("The Adapter Path", fontweight="bold")

plt.tight_layout()
plt.show()

print(f"Full projection update: {full_projection_params:,} trainable values")
print(
    f"LoRA rank-{adapter_rank}: A={adapter_a_params:,} + B={adapter_b_params:,} "
    f"= {adapter_total_params:,} ({adapter_percent:.2f}% of the full projection)"
)
print("This comparison measures adapter capacity, not peak memory, speed, or quality.")

### Optional Deep Dive: Crack Open the Trained Adapter

The core LoRA idea is already established: a frozen projection plus a small trainable correction. The next cells verify that claim inside the actual Riverside adapter by locating PEFT's $A$ and $B$ matrices, checking the rank limit, and tracing one correction through a real projection.

Skip to **Adapter Portability** if the measured parameter count and saved adapter are enough for your goal.

> **PyTorch → Keras:** `model.named_modules()` walks the full module tree so code can find every submodule PEFT injected (checking for a `lora_A` attribute), then reads the frozen `base_layer` and trainable `lora_A`/`lora_B` matrices directly off the object. **Keras/TF equivalent:** `model.layers` (recursively via nested `submodules`) walks the layer graph, and each layer's `layer.weights`/`get_weights()` exposes its tensors — Keras has no PEFT-style wrapper object, so there's no `lora_A`/`lora_B` attribute to introspect unless you built the adapter yourself as a custom layer.

In [ ]:
# Find every SmolLM2 projection wrapped by the continued-pretraining LoRA adapter.
lora_layers = [
    (name, module)
    for name, module in lora_pt_model.named_modules()
    if hasattr(module, "lora_A") and len(getattr(module, "lora_A")) > 0
]
expected_adapters = len(lora_pt_model.base_model.model.model.layers) * len(LORA_TARGET_MODULES)
print(
    f"PEFT wrapped {len(lora_layers)} projections; "
    f"expected {expected_adapters} for {len(LORA_TARGET_MODULES)} targets per decoder layer."
)
print("First wrapped module names:")
for name, _ in lora_layers[:8]:
    print(" ", name)

# Use the first query projection for concrete shape and rank introspection.
name0, layer0 = next(
    (name, module) for name, module in lora_layers if name.endswith("q_proj")
)
base0 = layer0.base_layer
lora_A0 = layer0.lora_A["default"]
lora_B0 = layer0.lora_B["default"]
scaling0 = layer0.scaling["default"]

print()
print(f"Inside {name0}:")
print(f"  Frozen q_proj:      {type(base0).__name__}, weight {tuple(base0.weight.shape)}")
print(f"  lora_A (down-proj): {tuple(lora_A0.weight.shape)}  <- trainable")
print(f"  lora_B (up-proj):   {tuple(lora_B0.weight.shape)}  <- trainable")
print(f"  scaling (alpha/r):  {scaling0}")
print(f"  Trained lora_B norm: {lora_B0.weight.norm().item():.4f}")

### Optional Check: Does the Update Really Have Rank at Most $r$?

Build PEFT's effective correction, $\Delta W=\text{scale}\cdot BA$, and inspect its singular values. Because $A$ passes through an $r$-wide bottleneck, no more than $r$ independent directions can remain non-zero. The next cell verifies that structural claim on the trained adapter.

> **PyTorch → Keras:** `layer.get_delta_weight(...)` (a PEFT method) computes the effective weight update `scaling · B @ A` as a plain tensor, and `torch.linalg.svdvals(...)` computes its singular values to verify the rank constraint. **Keras/TF equivalent:** `tf.linalg.svd(matrix, compute_uv=False)` computes singular values the same way; since Keras has no PEFT wrapper, you'd first need to manually multiply your own `B`/`A` weight matrices (`tf.matmul(B, A)`) to get the delta before taking its SVD.

In [ ]:
# Verify LoRA's rank constraint on the real trained adapter -- not asserted, measured via SVD.
delta_W = (
    layer0.get_delta_weight("default").detach().cpu()
)  # PEFT's own scaling * B @ A computation

# Singular values reveal how many independent directions ΔW actually has
singular_values = torch.linalg.svdvals(delta_W)
r = lora_A0.weight.shape[
    0
]  # the configured rank, read directly off the trained A matrix

# Only the first few dozen singular values are ever non-negligible for a rank-r update -- plotting
# all min(delta_W.shape) of them would squeeze the real cliff into an invisible sliver, so zoom in
# and use a log y-axis, which makes an 8-orders-of-magnitude drop actually visible.
n_show = min(30, len(singular_values))
floor = 1e-8  # log scale needs a positive floor; true near-zero values are clipped up for display only

# Clip near-zero singular values up to the floor so the log-scale axis can still plot them
plot_values = np.clip(singular_values[:n_show].numpy(), floor, None)

fig, ax = plt.subplots(figsize=(9, 4.5))

# Color the first r bars (real degrees of freedom) differently from the rest
ax.bar(
    range(n_show),
    plot_values,
    color=["mediumseagreen" if i < r else "lightgray" for i in range(n_show)],
)
ax.set_yscale("log")
ax.set_xlabel(f"Singular value index (first {n_show} of {len(singular_values)})")
ax.set_ylabel("Singular value magnitude (log scale)")
ax.set_title(
    f"Singular Value Spectrum of the Real Trained \u0394W = scaling \u00b7 B\u00b7A\n"
    f"({tuple(delta_W.shape)} matrix, configured rank r={r})",
    fontsize=11,
    fontweight="bold",
)
ax.legend(
    handles=[
        Patch(
            facecolor="mediumseagreen",
            edgecolor="black",
            label=f"First {r} singular values (the adapter's real degrees of freedom)",
        ),
        Patch(
            facecolor="lightgray",
            edgecolor="black",
            label=f"Indices {r + 1}-{n_show} (~0 by construction, floored for the log scale)",
        ),
    ],
    fontsize=8,
)
plt.tight_layout()
plt.show()

# Count how many singular values are meaningfully non-zero
nonzero = (singular_values > 1e-4).sum().item()
print(
    f"\u0394W shape: {tuple(delta_W.shape)} -> full rank would allow up to {min(delta_W.shape)} singular values"
)
print(f"Singular values above 1e-4: {nonzero} (matches the configured rank r={r})")
print(
    f"Largest singular value: {singular_values[0].item():.4f}   {r}th singular value: {singular_values[r - 1].item():.4f}"
)
print(
    f"First value past the rank cutoff (index {r}): {singular_values[r].item():.2e}  <- effectively zero"
)
print(
    "This is the real mechanics behind 'low-rank update': it isn't that training happened to find a "
    "low-rank \u0394W -- B(A(x))'s construction makes it mathematically impossible for \u0394W to have "
    "more than r independent directions, no matter what A and B learn."
)


### Optional Trace: See the Correction in One Forward Pass

Before training, PEFT initializes the adapter so it contributes no correction. After training, the combined projection differs slightly from the frozen base path. Forward hooks capture both outputs for one Riverside prompt, making the learned nudge visible without reimplementing PEFT internals.

> **PyTorch → Keras:** `module.register_forward_hook(...)` attaches a callback that captures a layer's output tensor during the forward pass, and the model call runs inside `torch.no_grad()` since no training is happening. **Keras/TF equivalent:** the closest analog is building an auxiliary `keras.Model` whose outputs include the intermediate layer(s) you want (`keras.Model(inputs=model.input, outputs=[layer.output, model.output])`), since Keras has no direct hook API; gradient tracking is simply avoided by not using a `tf.GradientTape()`.

In [ ]:
# Capture one real SmolLM2 query projection before and after its LoRA correction.
from matplotlib.patches import Patch

captured = {}


def make_hook(key):
    def hook(module, inputs, output):
        captured[key] = output.detach().cpu()
    return hook


hook_base = base0.register_forward_hook(make_hook("base_only"))
hook_combined = layer0.register_forward_hook(make_hook("combined"))
demo_prompt = STORY_SEED
enc_lora = tokenizer(demo_prompt, return_tensors="pt").to(device)
lora_pt_model.eval()
with torch.no_grad():
    _ = lora_pt_model(**enc_lora)
hook_base.remove()
hook_combined.remove()

base_out = captured["base_only"][0]
combined_out = captured["combined"][0]
lora_delta = combined_out - base_out
last_pos = base_out.shape[0] - 1
show_dims = min(60, base_out.shape[-1])

print(f"Raw continuation prompt: {demo_prompt!r}")
print(
    f"q_proj output shape: {tuple(base_out.shape)}; "
    "SmolLM2 uses separate q_proj/k_proj/v_proj/o_proj modules"
)
print()
print("At the last token position:")
print(f"  ||base q_proj output|| = {base_out[last_pos].norm().item():.3f}")
print(f"  ||LoRA delta||         = {lora_delta[last_pos].norm().item():.5f}")
print(
    "  delta / base norm     = "
    f"{(lora_delta[last_pos].norm() / base_out[last_pos].norm()).item():.4%}"
)

fig_static, (ax_static1, ax_static2) = plt.subplots(1, 2, figsize=(14, 4))
ax_static1.plot(
    base_out[last_pos, :show_dims].numpy(),
    color="steelblue",
    label="frozen q_proj",
)
ax_static1.plot(
    combined_out[last_pos, :show_dims].numpy(),
    color="coral",
    linestyle="--",
    label="q_proj + LoRA",
)
ax_static1.set_title(
    f"SmolLM2 Query Projection - first {show_dims} dimensions", fontsize=11
)
ax_static1.set_xlabel("q_proj output dimension")
ax_static1.legend(fontsize=8)
ax_static1.grid(alpha=0.3)

ax_static2.bar(
    np.arange(show_dims),
    lora_delta[last_pos, :show_dims].numpy(),
    color="mediumseagreen",
    label="LoRA delta",
)
ax_static2.set_title(
    f"LoRA delta at token position {last_pos}: scaling * B(A(x))", fontsize=11
)
ax_static2.set_xlabel("q_proj output dimension")
ax_static2.axhline(0, color="black", linewidth=0.8)
ax_static2.legend(fontsize=8)
ax_static2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

from matplotlib.animation import FuncAnimation
from IPython.display import HTML

n_tokens = lora_delta.shape[0]
delta_arr = lora_delta[:, :show_dims].numpy()
delta_max = np.abs(delta_arr).max() * 1.15 or 1e-6
fig_anim, ax_anim = plt.subplots(figsize=(12, 4))
bar_colors = ["mediumseagreen" if value >= 0 else "coral" for value in delta_arr[0]]
bars_anim = ax_anim.bar(np.arange(show_dims), delta_arr[0], color=bar_colors)
ax_anim.axhline(0, color="black", linewidth=0.8)
ax_anim.set_ylim(-delta_max, delta_max)
ax_anim.set_xlim(-1, show_dims)
ax_anim.set_xlabel(f"q_proj output dimension (first {show_dims})")
ax_anim.set_ylabel("LoRA delta")
tokens_decoded = tokenizer.convert_ids_to_tokens(enc_lora["input_ids"][0].tolist())
title_obj = ax_anim.set_title("")


def _update(frame):
    deltas = delta_arr[frame]
    for bar, value in zip(bars_anim, deltas):
        bar.set_height(value)
        bar.set_color("mediumseagreen" if value >= 0 else "coral")
    title_obj.set_text(
        f"SmolLM2 q_proj LoRA delta - token {frame}/{n_tokens - 1} "
        f"{tokens_decoded[frame]!r}; ||delta|| = {np.linalg.norm(deltas):.5f}"
    )
    return list(bars_anim) + [title_obj]


anim = FuncAnimation(fig_anim, _update, frames=n_tokens, interval=160, blit=False)
plt.close(fig_anim)
display(HTML(anim.to_jshtml(fps=6)))

### Adapter Portability: Swapping Onto a Compatible Base

LoRA adapters are swappable because they store small deltas rather than another complete base model. Compatibility is strict: the target module names, tensor dimensions, tokenizer contract, base model identifier, and selected hardware profile must agree.

The code below uses the profile-selected SmolLM2 instruction checkpoint and adapters regenerated by Parts 1 and 2 under the same profile directory. An adapter from the 135M, 360M, or 1.7B profile cannot be attached to either of the other sizes even though all three use Llama-style projection names.

This is why profile selection happens before checkpoint paths are resolved and why old artifacts are never overwritten or silently migrated.

> **PyTorch → Keras:** `PeftModel.from_pretrained(...)` attaches one adapter to a base model, `load_adapter(...)` registers a second adapter on the same base without reloading its weights, and `set_adapter(...)` switches which adapter's deltas are active — all cheap pointer/dict operations inside PEFT's routing layer. **Keras/TF equivalent:** no built-in adapter-swapping mechanism exists; the practical substitute is keeping separate fully fine-tuned (or separately frozen/unfrozen) model copies and calling `model.load_weights(...)` to switch between them, which is far more expensive than PEFT's adapter swap since it reloads full weight sets rather than a tiny delta.

In [ ]:
import gc
import os
from peft import PeftModel

swap_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
swap_model = PeftModel.from_pretrained(swap_base, CHECKPOINT_DIR / "peft-lora")
swap_model.eval()

adapter_kb = sum(
    os.path.getsize(os.path.join(CHECKPOINT_DIR / "peft-lora", filename))
    for filename in os.listdir(CHECKPOINT_DIR / "peft-lora")
) / 1024
base_mb = sum(
    parameter.numel() * parameter.element_size()
    for parameter in swap_base.parameters()
) / 1024**2
print(f"Base model in memory: {base_mb:.0f} MB (shared across adapters)")
print(f"Continued-pretraining adapter on disk: {adapter_kb:.0f} KB")
print()
print("[Continued-pretraining adapter]")
print(f"  {generate(swap_model, STORY_SEED, instruction=False)}")

swap_model.load_adapter(CHECKPOINT_DIR / "instruction-lora", adapter_name="instruct")
swap_model.set_adapter("instruct")
print()
print("[Instruction adapter]")
print(f"  {generate(swap_model, PROMPT)}")

swap_model.set_adapter("default")
print()
print("[Continued-pretraining adapter, restored]")
print(f"  {generate(swap_model, STORY_SEED, instruction=False)}")

q_proj_adapter = next(
    module
    for name, module in swap_model.named_modules()
    if name.endswith("q_proj")
    and hasattr(module, "lora_A")
    and "default" in module.lora_A
)
q_proj_A_shape = tuple(q_proj_adapter.lora_A["default"].weight.shape)
print()
print(f"q_proj LoRA A shape: {q_proj_A_shape}")
print(
    f"Compatibility contract: base={MODEL_NAME}, "
    f"hidden_size={swap_model.config.hidden_size}, targets={LORA_TARGET_MODULES}"
)
print(
    "Adapters from the previous architecture must be regenerated; "
    "checkpoint files are not converted in place."
)

del swap_base, swap_model
gc.collect()


## Concept 7: QLoRA - Make the Shared Base Fit

### The Riverside Bill

LoRA solved per-job update and artifact cost, but not resident base memory. On a much larger checkpoint, Riverside may be unable to fit even the frozen base beside activations and the adapter.

QLoRA combines a compact frozen base representation with floating-point LoRA matrices.

![One activation splitting into a frozen base path stored as four-bit codes with shared scales and reconstructed only for computation, and a trainable LoRA path whose correction rejoins the base output while gradients update only the adapter](images/qlora-quantized-base-lora-adapters.png)

Read the diagram as two paths through the same layer:

1. **Frozen base path:** compact codes and scale values represent the base weights in memory; the runtime reconstructs compute-friendly values for multiplication.
2. **Trainable adapter path:** LoRA $A$ and $B$ remain floating-point parameters and receive gradients.
3. **Addition:** the base output and adapter correction are added exactly as in ordinary LoRA.

The common four-bit format is **NF4**. The key intuition is that storage precision and arithmetic precision are separate choices: compact codes can represent the frozen base while computation uses a wider type.

```mermaid
flowchart LR
    X["Activation"] --> BASE["Reconstruct frozen base weights<br/>for this computation"]
    CODES["Compact frozen codes<br/>+ scales"] --> BASE
    X --> A["Trainable LoRA A"] --> B["Trainable LoRA B"]
    BASE --> ADD(("Add"))
    B --> ADD
    ADD --> Y["Layer output"]
```

During backpropagation, gradients pass through the base computation to earlier activations, but the frozen codes do not update. Optimizer state is needed only for the LoRA matrices.

### Make a Prediction

In the CPU analogy below, compact integer codes should reconstruct a compute-friendly base weight, the base should remain frozen, and gradients should appear only on the floating-point LoRA matrices.

### Inspect the Two Paths

The next cell uses a tiny uniform four-bit approximation because this notebook runs without a CUDA-specific QLoRA stack. It demonstrates compact codes, reconstruction, a floating-point adapter branch, and gradients landing only on the adapter. It is not NF4 or a trained QLoRA checkpoint.

For this small SmolLM2 profile, ordinary LoRA is the practical local path. QLoRA adds value only when frozen-base memory is the actual blocker.

The deeper kernel and format details belong in [Quantization in Depth](../../ai-infrastructure/06-quantization/quantization-in-depth.ipynb#appendix-b-nf4-and-qlora).

In [ ]:
# CPU-only structural analogy: uniform 4-bit codes, not bitsandbytes NF4.
torch.manual_seed(7)
in_features, out_features, rank = 4, 3, 2
alpha = 4

activation = torch.randn(2, in_features)
base_weight = torch.randn(out_features, in_features)  # frozen reference weights

# Approximate each output row with signed 4-bit codes and one scale.
scale = base_weight.abs().amax(dim=1, keepdim=True).clamp_min(1e-8) / 7
quantized_codes = torch.clamp(torch.round(base_weight / scale), -8, 7).to(torch.int8)
reconstructed_weight = quantized_codes.float() * scale

# Only these small floating-point matrices are trainable.
lora_a = torch.nn.Parameter(torch.randn(rank, in_features) * 0.05)
lora_b = torch.nn.Parameter(torch.randn(out_features, rank) * 0.05)

base_output = activation @ reconstructed_weight.T
adapter_output = (activation @ lora_a.T) @ lora_b.T * (alpha / rank)
layer_output = base_output + adapter_output
layer_output.square().mean().backward()

assert base_weight.grad is None
assert lora_a.grad is not None and lora_b.grad is not None
print("INSPECT THE QLORA TWO-PATH ANALOGY")
print(f"Stored frozen-base codes: {quantized_codes.dtype}")
print(f"Reconstructed compute weights: {reconstructed_weight.dtype}")
print(f"Frozen base gradient: {base_weight.grad}")
print(f"LoRA gradient norms: A={lora_a.grad.norm():.4f}, B={lora_b.grad.norm():.4f}")
print(f"Toy reconstruction MAE: {(base_weight - reconstructed_weight).abs().mean():.4f}")
print("Riverside takeaway: compact base storage and trainable job adapters are separate paths.")


### Riverside Synthesis - Which Bill Did Each Strategy Remove?

Return to the question that opened the notebook: what must Riverside pay for each adaptation? The trained objects support two direct observations: how many values receive updates and how much gradient-plus-Adam state those values imply under an fp32 proxy.

Predict the ordering before running the chart: full fine-tuning should carry the largest update-state bill, partial freezing should reduce it, and LoRA should reduce it dramatically. QLoRA is omitted because its additional value is frozen-base residency, not fewer adapter parameters.

> **PyTorch → Keras:** PyTorch counts trainable values with `sum(p.numel() for p in model.parameters() if p.requires_grad)`. The Keras equivalent is `sum(np.prod(weight.shape) for weight in model.trainable_weights)`. In either framework, parameter counts support an update-state estimate only; they do not measure peak memory or speed.

In [ ]:
# Compare measured trainable counts and an explicitly bounded update-state proxy.
total_params = sum(parameter.numel() for parameter in base_model.parameters())
full_ft_params = total_params
partial_ft_params = sum(
    parameter.numel() for parameter in freeze_model.parameters() if parameter.requires_grad
)
lora_params = sum(
    parameter.numel() for parameter in lora_pt_model.parameters() if parameter.requires_grad
)

techniques = ["Full fine-tuning", "Partial freezing", "LoRA (r=8)"]
param_counts = [full_ft_params, partial_ft_params, lora_params]
param_pcts = [count / total_params * 100 for count in param_counts]
colors = ["steelblue", "coral", "mediumseagreen"]

# fp32 gradient (4 bytes) + two fp32 Adam moments (8 bytes) per trainable parameter.
# Model weights, activations, temporary buffers, and allocator overhead are deliberately excluded.
update_state_bytes_per_param = 12
update_state_mb = [
    count * update_state_bytes_per_param / 1024**2 for count in param_counts
]

fig, (count_axis, state_axis) = plt.subplots(1, 2, figsize=(14, 5.5))

count_bars = count_axis.bar(techniques, param_counts, color=colors, edgecolor="black")
count_axis.set_yscale("log")
count_axis.set_ylabel("Trainable parameters (log scale)")
count_axis.set_title("Measured Trainable Parameter Count", fontweight="bold")
count_axis.grid(axis="y", alpha=0.25)
for bar, count, percent in zip(count_bars, param_counts, param_pcts):
    count_axis.annotate(
        f"{count:,}\n{percent:.2f}%",
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
    )

state_bars = state_axis.barh(techniques, update_state_mb, color=colors, edgecolor="black")
state_axis.invert_yaxis()
state_axis.set_xscale("log")
state_axis.set_xlabel("Estimated fp32 gradient + Adam state (MiB, log scale)")
state_axis.set_title("Trainable-State Proxy, Not Peak Memory", fontweight="bold")
state_axis.grid(axis="x", alpha=0.25)
for bar, state_mb in zip(state_bars, update_state_mb):
    state_axis.annotate(
        f"{state_mb:,.1f} MiB",
        xy=(bar.get_width(), bar.get_y() + bar.get_height() / 2),
        xytext=(6, 0),
        textcoords="offset points",
        ha="left",
        va="center",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

print(f"{'Technique':<22} {'Trainable':>14} {'Percent':>10} {'Grad+Adam proxy':>18}")
print("-" * 68)
for technique, count, percent, state_mb in zip(
    techniques, param_counts, param_pcts, update_state_mb
):
    print(f"{technique:<22} {count:>14,} {percent:>9.2f}% {state_mb:>15,.1f} MiB")

print("\nBoundary: this proxy excludes resident weights, activations, temporary buffers, and hardware effects.")
print("Trainable-parameter ratios do not establish peak memory, elapsed time, throughput, or quality.")
print("\nRIVERSIDE TAKEAWAY")
print("Full FT: maximum freedom, complete per-job model.")
print("Partial freezing: smaller update state, still a complete per-job model.")
print("LoRA: one shared base plus a small per-job adapter.")
print("QLoRA: use when the shared frozen base itself is the memory blocker.")

### Optional Storage Exercise: Dynamic Int8 Conversion

Post-training dynamic int8 conversion is not QLoRA training. It is included only to make one storage distinction tangible: a trained floating-point checkpoint can be converted into a different CPU-oriented representation after training.

The exercise below loads one full checkpoint, converts supported linear layers, and compares serialized parameter-state size. It does not generate text or interpret model behavior. Part 3 owns all evaluation.

For the wider method catalog, see [Quantization in Depth](../../ai-infrastructure/06-quantization/quantization-in-depth.ipynb).

> **PyTorch → Keras:** `torch.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)` converts supported linear weights to int8 for CPU inference. TensorFlow's closest equivalent uses the TFLite converter with `tf.lite.Optimize.DEFAULT`, producing a separate `.tflite` artifact rather than changing the in-memory Keras model.

In [ ]:
# CPU-native post-training conversion: inspect representation and serialized size only.
import io

full_checkpoint_dir = CHECKPOINT_DIR / "non-instruction-full"
config_path = full_checkpoint_dir / "config.json"
fp32_model = None
int8_model = None

if not config_path.is_file():
    print(
        "SKIP: the optional int8 storage exercise needs the Part 1 continued-pretraining "
        f"checkpoint at {full_checkpoint_dir}. Run Part 1 first."
    )
else:
    full_checkpoint_config = json.loads(config_path.read_text(encoding="utf-8"))
    if full_checkpoint_config.get("model_type") != base_model.config.model_type:
        raise RuntimeError(
            "The continued-pretraining checkpoint uses an incompatible architecture. "
            "Regenerate Part 1 checkpoints for this model profile."
        )
    fp32_model = AutoModelForCausalLM.from_pretrained(full_checkpoint_dir)
    fp32_model.eval()
    int8_model = torch.quantization.quantize_dynamic(
        fp32_model, {torch.nn.Linear}, dtype=torch.qint8
    )


def state_dict_size_mb(model):
    """Measure serialized parameter-state size without writing a temporary artifact."""
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return buffer.getbuffer().nbytes / 1e6


if fp32_model is not None and int8_model is not None:
    fp32_mb = state_dict_size_mb(fp32_model)
    int8_mb = state_dict_size_mb(int8_model)
    print(f"fp32 state_dict size: {fp32_mb:.1f} MB")
    print(f"int8 state_dict size: {int8_mb:.1f} MB")
    print(f"Serialized state reduction: {(1 - int8_mb / fp32_mb) * 100:.1f}%")

    quantized_linear_count = sum(
        "quantized" in type(module).__module__ and type(module).__name__ == "Linear"
        for module in int8_model.modules()
    )
    print(f"Dynamically quantized linear modules: {quantized_linear_count}")
    print("Part 3 owns runtime and behavioral evaluation of converted artifacts.")

---

## Riverside Takeaway: Choose the Smallest Mechanism That Fits the Job

The parameter mechanisms are now visible:

| Strategy | What receives gradients | What the adaptation saves |
| --- | --- | --- |
| Full fine-tuning | Every trainable model weight | A complete changed checkpoint |
| Partial freezing | A selected layer slice | A complete changed checkpoint |
| LoRA | Small $A$ and $B$ matrices beside frozen projections | A compact adapter plus a pinned base dependency |
| QLoRA path | Floating-point LoRA matrices beside a compact frozen base | Adapter plus a compatible low-bit base/runtime contract |

Riverside does not win by choosing the fanciest technique. It wins by removing the bill that actually blocks the product:

- choose full fine-tuning when maximum update freedom justifies a complete model;
- choose partial freezing when a known layer slice is enough but full artifacts are acceptable;
- choose LoRA when many Riverside jobs should share one base; and
- choose QLoRA when that shared frozen base no longer fits comfortably in memory.

Continue to **[Part 3: Evaluation, Comparison & Decision](03-llm-finetuning-comparison-and-decision.ipynb)**, where evaluation begins.